<a href="https://colab.research.google.com/github/efitznm/Capstone-Colab/blob/main/Capstone_2025_AHAR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Sources

[2025 AHAR: Part 1 - PIT Estimates of Homelessness in the U.S.
](https://www.huduser.gov/portal/datasets/ahar/2025-ahar-part-1-pit-estimates-of-homelessness-in-the-us.html)

# Potential Business Questions

1. Which states and Continuums of Care (CoCs) have the greatest housing capacity?
2. Which geographic areas have the fewest available beds relative to their homeless populations?
3. Are rural communities underserved compared with urban communities?
4. How does New Mexico compare with other states in housing inventory?
5. Are certain regions more dependent on seasonal shelters?

# Load Modules and Functions

In [ ]:
# Define the metadata tools
%%writefile metadata01.py
import pandas as pd

def metadata( dataframe ):
  '''Given a dataframe, returns a dataframe of metadata about the dataframe'''
  metadata_df = pd.DataFrame()
  metadata_df["Data_types"] = dataframe.dtypes
  metadata_df["Count"] = len(dataframe)
  metadata_df["Nulls"] = dataframe.isnull().sum()
  metadata_df["Nulls_pct"] = ( metadata_df["Nulls"] / metadata_df["Count"] * 100 ).round(1)
  metadata_df["Memory"] = dataframe.memory_usage( deep = True)
  metadata_df["NUnique"] = dataframe.nunique()
  metadata_df["NUnique_pct"] = (dataframe.nunique() / metadata_df["Count"] * 100).round(1)
  metadata_df = metadata_df.join( dataframe.describe( include = "all" ).transpose() )
  metadata_df = metadata_df.astype( { "count" : int } )
  if dataframe.select_dtypes(include=['number']).shape[1] :
    metadata_df["IRQ"] = metadata_df["75%"] - metadata_df["25%"]
    metadata_df["range"] = metadata_df["max"] - metadata_df["min"]
    metadata_df["sum"] = metadata_df["mean"] * metadata_df["count"]
    metadata_df = (
      metadata_df
      .rename( columns = {
        "25%" : "Q1_25%",
        "50%" : "Q2_median",
        "75%" : "Q3_75%",
        }
      )
    )
  return metadata_df

def cols_to_drop( dataframe ):
  '''Given a dataframe, returns columns that should likely be dropped'''
  md = metadata( dataframe )
  filter = md["Nulls_pct"] >= 40
  return md[ filter ]["Nulls_pct"].to_dict()

def likely_ids( dataframe ):
  '''Given a dataframe, returns a dictionary of likely ID columns'''
  md = metadata( dataframe )
  return md["NUnique_pct"][ md["NUnique_pct"] > 95 ].sort_values( ascending = False )

Overwriting metadata01.py


In [ ]:
# Import libraries
import pandas as pd
import requests
from io import BytesIO
import metadata01 as md01

# Load the 2025 Housing Inventory Counts-by-State Data Set

In [ ]:
# Define the URL for our dataset
url = "https://www.huduser.gov/portal/sites/default/files/xls/2025-HIC-Counts-by-State.csv"

In [ ]:
# Define headers (so we can pretend to be a browser)
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/csv,*/*"
}

In [ ]:
# Use the headers in the get request
response = requests.get(url, headers=headers)

print("Status:", response.status_code)
print("Content-Type:", response.headers.get("Content-Type"))
print("Content size:", len(response.content))

Status: 200
Content-Type: application/octet-stream
Content size: 11334278


In [ ]:
# Read in the data with pandas
df = pd.read_csv(BytesIO(response.content))

/tmp/ipykernel_8612/2145311823.py:2: DtypeWarning: Columns (90) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(BytesIO(response.content))


# Summary Statistics

In [ ]:
# See the first 5 rows
df.head()

,Row #,CocState,CoC,HudNum,Status,Year,Organization Name,HMIS Org ID,Project Name,Common Name,...,Total Seasonal Beds,Availability Start Date,Availability End Date,O/V Beds,Pit Count,Total Beds,Updated On,sandyRelated,sandyRelatedNote,HOPWAMedAssistedLivingFac
0,1,WY,Wyoming Statewide CoC,WY-500,Submitted,2025,COMEA1 - Cooperative Ministry for Emergency As...,7,COMEA - ESG - Emergency Shelter - Cheyenne,NaN,...,0.0,NaN,NaN,0.0,60.0,122,10-Jun-25,No,NaN,2.0
1,2,WY,Wyoming Statewide CoC,WY-500,Submitted,2025,CCS1 - Council of Community Services - Gillette,34,CCS - Way Station 1 - ESG - Emergency Shelter ...,NaN,...,0.0,NaN,NaN,0.0,40.0,51,10-Jun-25,No,NaN,2.0
2,3,WY,Wyoming Statewide CoC,WY-500,Submitted,2025,HIC WRM1 - Wyoming Rescue Mission - Casper,36,HIC WRM - Men Dorm - NA - Emergency Shelter - ...,NaN,...,0.0,NaN,NaN,0.0,36.0,68,30-May-25,No,NaN,2.0
3,4,WY,Wyoming Statewide CoC,WY-500,Submitted,2025,CCS1 - Council of Community Services - Gillette,34,CCS - Way Station 3 - CoC - Permanent Supporti...,NaN,...,0.0,NaN,NaN,0.0,7.0,8,29-May-25,No,NaN,2.0
4,5,WY,Wyoming Statewide CoC,WY-500,Submitted,2025,CALC1- Community Action Laramie County - Cheyenne,160,CALC - VA/GPD - Bridge Housing - Cheyenne,NaN,...,0.0,NaN,NaN,0.0,8.0,10,29-May-25,No,NaN,2.0


In [ ]:
# See some metadata
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30012 entries, 0 to 30011
Data columns (total 92 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Row #                         30012 non-null  int64  
 1   CocState                      30012 non-null  object 
 2   CoC                           30012 non-null  object 
 3   HudNum                        30012 non-null  object 
 4   Status                        30012 non-null  object 
 5   Year                          30012 non-null  int64  
 6   Organization Name             30012 non-null  object 
 7   HMIS Org ID                   30012 non-null  object 
 8   Project Name                  30010 non-null  object 
 9   Common Name                   3032 non-null   object 
 10  HMIS Project ID               30012 non-null  object 
 11  HIC Date                      30012 non-null  object 
 12  Project Type                  30012 non-null  object 
 13  B

In [ ]:
# Number of rows and columns
df.shape

(30012, 92)

In [ ]:
# Summary statistics
df.describe().transpose()

,count,mean,std,min,25%,50%,75%,max
Row #,30012.0,134.959483,210.273295,1.0,22.0,58.0,151.0,1322.0
Year,30012.0,2024.999534,0.021593,2024.0,2025.0,2025.0,2025.0,2025.0
mcKinneyVentoEsgEs,30012.0,0.069439,0.254203,0.0,0.0,0.0,0.0,1.0
mcKinneyVentoEsgRrh,30012.0,0.031554,0.174812,0.0,0.0,0.0,0.0,1.0
mcKinneyVentoEsgCov,30012.0,0.006597,0.080957,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...
Total Seasonal Beds,30008.0,0.718708,8.171139,0.0,0.0,0.0,0.0,550.0
O/V Beds,30004.0,2.113318,113.750283,0.0,0.0,0.0,0.0,19430.0
Pit Count,29992.0,37.402741,159.940809,0.0,6.0,15.0,36.0,19430.0
Total Beds,30012.0,44.093463,168.223553,0.0,8.0,20.0,43.0,19430.0


In [ ]:
# Use our function to see all the metadata
md01.metadata(df)

,Data_types,Count,Nulls,Nulls_pct,Memory,NUnique,NUnique_pct,count,unique,top,...,mean,std,min,Q1_25%,Q2_median,Q3_75%,max,IRQ,range,sum
Row #,int64,30012,0,0.0,240096,1322,4.4,30012,NaN,NaN,...,134.959483,210.273295,1.0,22.0,58.0,151.0,1322.0,129.0,1321.0,4050404.0
CocState,object,30012,0,0.0,1530612,54,0.2,30012,54,CA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
CoC,object,30012,0,0.0,2358713,386,1.3,30012,386,New York City CoC,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
HudNum,object,30012,0,0.0,1650660,386,1.3,30012,386,NY-600,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Status,object,30012,0,0.0,1740696,1,0.0,30012,1,Submitted,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Total Beds,int64,30012,0,0.0,240096,599,2.0,30012,NaN,NaN,...,44.093463,168.223553,0.0,8.0,20.0,43.0,19430.0,35.0,19430.0,1323333.0
Updated On,object,30012,0,0.0,1733281,71,0.2,30012,71,12-Jun-25,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sandyRelated,object,30012,0,0.0,1530642,2,0.0,30012,2,No,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sandyRelatedNote,object,30012,29982,99.9,961776,5,0.0,30,5,Helene September 2024,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# See where there are nulls
df.isnull().sum().sort_values(ascending=False)

,0
sandyRelatedNote,29982
Availability Start Date,29451
Availability End Date,29451
Operating End Date,28663
address2,27506
...,...
Year-Round Beds,0
CH Beds HH w only Children,0
Total Beds,0
sandyRelated,0


In [ ]:
# See columns to consider dropping
md01.cols_to_drop(df)

{'Common Name': 89.9,
 'Bed Type': 68.6,
 'Target Population': 89.3,
 'Operating End Date': 95.5,
 'federalFundingOtherSpecify': 74.5,
 'address2': 91.7,
 'Availability Start Date': 98.1,
 'Availability End Date': 98.1,
 'sandyRelatedNote': 99.9}

In [ ]:
# See identifier columns
md01.likely_ids(df)

,NUnique_pct
Project Name,97.6


In [ ]:
# Create a list of all our column names
df.columns.tolist()

['Row #',
 'CocState',
 'CoC',
 'HudNum',
 'Status',
 'Year',
 'Organization Name',
 'HMIS Org ID',
 'Project Name',
 'Common Name',
 'HMIS Project ID',
 'HIC Date',
 'Project Type',
 'Bed Type',
 'Geocode',
 'HMIS Participating',
 'Inventory Type',
 'Inventory Start Date',
 'Target Population',
 'Operating Start Date',
 'Operating End Date',
 'mcKinneyVentoEsgEs',
 'mcKinneyVentoEsgRrh',
 'mcKinneyVentoEsgCov',
 'mcKinneyVentoEsgRUSH',
 'mcKinneyVentoCocSh',
 'mcKinneyVentoCocTh',
 'mcKinneyVentoCocPsh',
 'mcKinneyVentoCocRrh',
 'mcKinneyVentoCocSro',
 'mcKinneyVentoCocThRrh',
 'mcKinneyVentoSpC',
 'mcKinneyVentoS8',
 'mcKinneyVentoShp',
 'mcKinneyVentoYhdp',
 'mcKinneyVentoYhdpRenewals',
 'mcKinneyVentoUnshelt',
 'mcKinneyVentoRural',
 'federalFundingVash',
 'federalFundingSsvf',
 'federalFundingGpdBh',
 'federalFundingGpdLd',
 'federalFundingGpdHh',
 'federalFundingGpdCt',
 'federalFundingGpdSith',
 'federalFundingGpdTp',
 'federalFundingHchvCrs',
 'federalFundingHchvSh',
 'federalF

# Data Frame of all NM related Housing Programs

In [ ]:
# Make a list of all columns with rows containing "NM" or "New Mexico"
nm_columns = df.columns[
    df.astype(str).apply(
        lambda col: col.str.contains('NM|New Mexico', case=False).any()
    )
].tolist()

nm_columns

['CocState',
 'CoC',
 'HudNum',
 'Organization Name',
 'HMIS Org ID',
 'Project Name',
 'Common Name',
 'federalFundingOtherSpecify',
 'address1',
 'city',
 'state']

In [ ]:
# Create a dictionary of those column names (keys) and their values
nm_dict = {
    col: df[col].tolist()
    for col in nm_columns
}

nm_dict

{'CocState': ['WY',
  'WY',
  'WY',
  'WY',
  'WY',
  'WY',
  'WY',
  'WY',
  'WY',
  'WY',
  'WY',
  'WY',
  'WY',
  'WY',
  'WY',
  'WY',
  'WY',
  'WY',
  'WY',
  'WY',
  'WY',
  'WY',
  'WY',
  'WY',
  'WY',
  'WY',
  'WY',
  'WY',
  'WY',
  'WY',
  'WY',
  'WY',
  'WY',
  'WY',
  'WY',
  'WY',
  'WY',
  'WY',
  'WY',
  'WY',
  'WY',
  'WY',
  'WY',
  'WY',
  'WY',
  'WY',
  'WY',
  'WY',
  'WY',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'WV',
  'W

In [ ]:
# List of housing-related columns
housing_cols = [
    'Project Type',
    'Housing Type',
    'Bed Type',
    'Inventory Type',
    'Target Population',
    'Total Beds',
    'Year-Round Beds',
    'Total Seasonal Beds',
    'Pit Count',
    'mcKinneyVentoCocSh',
    'mcKinneyVentoCocTh',
    'mcKinneyVentoCocPsh',
    'mcKinneyVentoCocRrh',
    'mcKinneyVentoEsgEs',
    'mcKinneyVentoEsgRrh'
]

In [ ]:
# Combine the original dictionary of NM related columns and housing-related columns
selected_cols = list(set(nm_dict.keys()).union(housing_cols))

# Create a new DataFrame
nm_housing_df = df[selected_cols].copy()
nm_housing_df.head()

,CoC,Pit Count,Target Population,mcKinneyVentoCocTh,address1,HudNum,Common Name,Housing Type,Total Seasonal Beds,Organization Name,...,mcKinneyVentoEsgRrh,mcKinneyVentoCocRrh,Year-Round Beds,Project Name,Inventory Type,CocState,state,mcKinneyVentoCocPsh,Bed Type,mcKinneyVentoCocSh
0,Wyoming Statewide CoC,60.0,NaN,0,1504 Stinson Ave,WY-500,NaN,SB-S,0.0,COMEA1 - Cooperative Ministry for Emergency As...,...,0,0,122,COMEA - ESG - Emergency Shelter - Cheyenne,C,WY,WY,0,F,0
1,Wyoming Statewide CoC,40.0,NaN,0,114 4-J RD,WY-500,NaN,SB-S,0.0,CCS1 - Council of Community Services - Gillette,...,0,0,51,CCS - Way Station 1 - ESG - Emergency Shelter ...,C,WY,WY,0,F,0
2,Wyoming Statewide CoC,36.0,NaN,0,230 N park St & 132 N kimble Street,WY-500,NaN,SB-S,0.0,HIC WRM1 - Wyoming Rescue Mission - Casper,...,0,0,68,HIC WRM - Men Dorm - NA - Emergency Shelter - ...,C,WY,WY,0,F,0
3,Wyoming Statewide CoC,7.0,NaN,0,114 4-J RD,WY-500,NaN,SB-S,0.0,CCS1 - Council of Community Services - Gillette,...,0,0,8,CCS - Way Station 3 - CoC - Permanent Supporti...,C,WY,WY,1,NaN,0
4,Wyoming Statewide CoC,8.0,NaN,0,"200 W 17th Street, Suite 210",WY-500,NaN,SB-S,0.0,CALC1- Community Action Laramie County - Cheyenne,...,0,0,10,CALC - VA/GPD - Bridge Housing - Cheyenne,C,WY,WY,0,NaN,0


# Backup and Restore

In [ ]:
# Backup
nm_housing_df_bak = nm_housing_df.copy()

In [ ]:
# Restore from backup
nm_housing_df = nm_housing_df_bak.copy()

# Filter the NM Housing Data Frame

## Housing Categories

Emergency Shelter, Transitional Housing, Rapid Re-Housing, Permanent Supportive Housing, Safe Haven

What do the codes mean in 'project type' and 'housing type'?

In [ ]:
# Primary columns for identifying housing categories (see project and housing type -- helpful to have a data dictionary or SME here)
housing_category_cols = [
    'Project Type',
    'Housing Type',
    'Bed Type',
    'Inventory Type',
    'Target Population'
]

nm_housing_categories = nm_housing_df[housing_category_cols].copy()

nm_housing_categories.head()

,Project Type,Housing Type,Bed Type,Inventory Type,Target Population
0,ES,SB-S,F,C,NaN
1,ES,SB-S,F,C,NaN
2,ES,SB-S,F,C,NaN
3,PSH,SB-S,NaN,C,NaN
4,TH,SB-S,NaN,C,NaN


In [ ]:
nm_housing_categories['Housing Type'].value_counts()

,count
Housing Type,
SB-S,15425
TB,11750
SB-C,2837


## McKinney-Vento

In [ ]:
# McKinney-Vento funding and program columns
mckinney_cols = [
    'mcKinneyVentoCocSh',
    'mcKinneyVentoCocTh',
    'mcKinneyVentoCocPsh',
    'mcKinneyVentoCocRrh',
    'mcKinneyVentoEsgEs',
    'mcKinneyVentoEsgRrh'
]

nm_mckinney = nm_housing_df[mckinney_cols].copy()

nm_mckinney.head()

,mcKinneyVentoCocSh,mcKinneyVentoCocTh,mcKinneyVentoCocPsh,mcKinneyVentoCocRrh,mcKinneyVentoEsgEs,mcKinneyVentoEsgRrh
0,0,0,0,0,1,0
1,0,0,0,0,1,0
2,0,0,0,0,0,0
3,0,0,1,0,0,0
4,0,0,0,0,0,0


In [ ]:
nm_mckinney.sum(numeric_only=True)

,0
mcKinneyVentoCocSh,38
mcKinneyVentoCocTh,438
mcKinneyVentoCocPsh,3951
mcKinneyVentoCocRrh,1314
mcKinneyVentoEsgEs,2084
mcKinneyVentoEsgRrh,947


# Housing Capactiy

Hold off on this

In [ ]:
housing_capacity_cols = [
    'Total Beds',
    'Year-Round Beds',
    'Total Seasonal Beds',
    'Pit Count',
    'Beds HH w/ Children',
    'Beds HH w/o Children',
    'Beds HH w/ only Children',
    'Units HH w/ Children'
]

In [ ]:
missing_cols = [
    col for col in housing_capacity_cols
    if col not in nm_housing_df.columns
]

missing_cols

['Beds HH w/ Children',
 'Beds HH w/o Children',
 'Beds HH w/ only Children',
 'Units HH w/ Children']

In [ ]:
nm_housing_capacity = df.loc[
    df['CocState'] == 'NM',
    housing_capacity_cols
].copy()

nm_housing_capacity.head()

,Total Beds,Year-Round Beds,Total Seasonal Beds,Pit Count,Beds HH w/ Children,Beds HH w/o Children,Beds HH w/ only Children,Units HH w/ Children
11157,12,12,0.0,12.0,11.0,1.0,0.0,1.0
11158,32,32,0.0,40.0,28.0,4.0,0.0,12.0
11159,5,5,0.0,3.0,0.0,2.0,3.0,0.0
11160,22,22,0.0,1.0,11.0,11.0,0.0,2.0
11161,115,115,0.0,115.0,97.0,18.0,0.0,31.0


In [ ]:
nm_housing_capacity.describe()

,Total Beds,Year-Round Beds,Total Seasonal Beds,Pit Count,Beds HH w/ Children,Beds HH w/o Children,Beds HH w/ only Children,Units HH w/ Children
count,185.000000,185.000000,185.000000,185.000000,185.000000,185.000000,185.000000,185.000000
mean,49.383784,48.308108,0.621622,40.313514,24.827027,22.194595,1.286486,7.297297
std,85.286338,85.038592,8.454968,81.748149,44.402095,57.085986,5.851336,11.491171
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,12.000000,12.000000,0.000000,6.000000,0.000000,0.000000,0.000000,0.000000
50%,22.000000,22.000000,0.000000,18.000000,10.000000,6.000000,0.000000,3.000000
75%,50.000000,48.000000,0.000000,40.000000,24.000000,20.000000,0.000000,9.000000
max,600.000000,600.000000,115.000000,762.000000,280.000000,520.000000,70.000000,70.000000
